# Retrieval-Augmented Generation & Vector DBs
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Dense vs Sparse Retrieval, HNSW, FAISS, Chunking strategies, Re-ranking with Cross-Encoders, and HyDE.

---

### 📖 Conceptual Foundations

#### What is Retrieval-Augmented Generation (Lewis et al., 2020)?

Parametric knowledge stored in LLM weights suffers from knowledge cutoffs, hallucination of non-existent facts, and inability to access private proprietary data. **RAG** resolves this by pairing an information retrieval system with a generative model:
1. **Retrieve:** Given user query $q$, search external knowledge bases to find the top-$k$ most relevant text chunks $\{d_1, \dots, d_k\}$.
2. **Augment:** Construct a context-rich prompt: $\text{Prompt} = [\text{SystemInstructions}, d_1, \dots, d_k, q]$.
3. **Generate:** The LLM generates an answer grounded strictly on the retrieved context.


#### Retrieval Paradigms: Sparse vs Dense vs Hybrid

| Retriever Type | Underlying Mechanism | Strengths | Weaknesses | Examples |
| --- | --- | --- | --- | --- |
| Sparse (Lexical) | Exact keyword matching via inverted indices and Term Frequency-Inverse Document Frequency. | High precision on domain-specific codes, part numbers, exact acronyms, and product IDs. | Vulnerable to vocabulary mismatch (synonyms: 'automobile' vs 'car'). | BM25, TF-IDF, Elasticsearch, SPLADE. |
| Dense (Semantic) | Deep bi-encoders project text into continuous semantic vector embeddings; computes cosine/dot-product distance. | Understands conceptual intent, synonyms, and multilingual semantics. | Can miss exact alphanumeric part numbers; computationally heavier. | OpenAI text-embedding-3, BGE, E5, Cohere Embed. |
| Hybrid Retrieval | Combines sparse (BM25) and dense retrieval scores using **Reciprocal Rank Fusion (RRF)**. | Best of both worlds: robust keyword matching and deep semantic understanding. | Requires managing two search indices concurrently. | Pinecone Hybrid, Weaviate, Qdrant, Elasticsearch. |


#### Reciprocal Rank Fusion (RRF) Formula

Given rankings from multiple retrievers $R \in \mathcal{R}$ for document $d$, RRF calculates a combined score:

$$\text{RRF}(d) = \sum_{R \in \mathcal{R}} \frac{1}{k + \text{rank}_R(d)}$$

Where $k$ is a constant (typically $k=60$) that prevents top-ranked items from drowning out consistent runners-up across systems.


#### Vector Indexing Algorithms: Flat vs IVF vs HNSW

- **Flat Index:** Exact brute-force $O(N)$ comparison against all $N$ vectors. 100% recall, but completely unscalable beyond $100,000$ vectors.
- **Inverted File (IVF):** Partitions vector space into Voronoi cells via k-means. Queries search only the nearest centroids and their associated inverted lists.
- **Hierarchical Navigable Small World (HNSW, Malkov & Yashunin, 2018):** State-of-the-art multi-layer proximity graph. Queries enter at a sparse top layer and greedily zoom into fine-grained local neighborhoods in $O(\log N)$ time.


#### Two-Stage Retrieval with Cross-Encoder Re-Ranking

1. **Stage 1 (Bi-Encoder Retrieval):** Fast vector search retrieves candidate pool of top-50 documents in $<15\text{ms}$.
2. **Stage 2 (Cross-Encoder Re-Ranker):** Concatenates $[\text{query}, d_i]$ and passes it through a deep bidirectional Transformer with full cross-attention. Ranks top-50 candidates with high precision down to the top-5 documents fed to the LLM.



In [ ]:
# %matplotlib inline
import numpy as np

# Simulated embedding function (producing normalized 4D vectors)
def mock_embed(text):
    np.random.seed(abs(hash(text)) % (2**32))
    v = np.random.randn(4)
    return v / np.linalg.norm(v)

# Knowledge Base documents
corpus = [
    'FastAPI is a modern, fast web framework for building APIs with Python 3.8+.',
    'Convolutional neural networks excel at extracting 2D spatial features from images.',
    'Retrieval-Augmented Generation combines external document retrieval with LLMs to reduce hallucination.',
    'Gradient descent updates weights in the opposite direction of the loss gradient.'
]

# Build vector index
index = np.array([mock_embed(doc) for doc in corpus])

# Query
query = 'How to stop LLM hallucinations using external documents?'
q_vec = mock_embed(query)

# Cosine similarity search (dot product of unit vectors)
similarities = np.dot(index, q_vec)
top_idx = np.argmax(similarities)

print(f'Query: "{query}"')
print(f'Top Match Score: {similarities[top_idx]:.4f}')
print(f'Retrieved Document: "{corpus[top_idx]}"')

# Construct Grounded Prompt
prompt = f'''Context:\n{corpus[top_idx]}\n\nQuestion: {query}\nAnswer based ONLY on the context above:'''
print('\n=== Constructed Grounded Prompt ===\n' + prompt)


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

